#NaijaCode RAG 助手🇳🇬

第 5 周练习 — 构建 RAG（检索增强生成）助手

这是 **NaijaCode**（一家虚构的尼日利亚科技公司）的知识工作者聊天机器人。
它使用 RAG 来回答有关公司员工、产品和政策的问题。

我们正在做什么：
1. 从我们的知识库加载文档
2. 将它们分成块
3. 使用 HuggingFace 创建向量嵌入
4. 存储在 Chroma 矢量数据库中
5. 使用 Gradio UI 构建 RAG 聊天

作者：**维克多征服者** 🚀

In [ ]:
# 数据设置 - 下载知识库（如果缺少）
# Data Setup — Download knowledge base if missing
import os
import zipfile
import urllib.request

KB_DIR = "knowledge-base"
ZIP_URL = "https://drive.google.com/uc?export=download&id=1VoCSN31onJCiYfDlxOH2PL5YQyngN7Sh"
ZIP_FILE = "knowledge-base.zip"

if not os.path.exists(KB_DIR):
    print(f"Knowledge base not found. Downloading...")
    try:
        urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)
        # 如果目录不存在则创建
        # Create the directory if it doesn't exist
        os.makedirs(KB_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extractall(KB_DIR)
        os.remove(ZIP_FILE)
        print("Knowledge base downloaded and extracted!")
    except Exception as e:
        print(f"Error downloading data: {e}")
        print("Please ensure you have an internet connection and the link is valid.")
else:
    print("Knowledge base already exists.")

In [ ]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# 设置
# Setup

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — oya go add am to your .env!")

MODEL = "gpt-4o-mini"
DB_NAME = "naijacode_vector_db"

## 第 1 步：加载知识库

我们有关于以下内容的 Markdown 文件：
- **员工/** — Adaeze (首席技术官)、Chinedu (后端主管)、Fatima (产品主管)、Seun (首席执行官)、Amina (前端主管)
- **产品/** — CodeNaija IDE、NaijaCode 市场、NaijaCode AI
- **公司/** — 公司概况、政策和福利

In [ ]:
# 从知识库加载所有文档
# Load all documents from the knowledge base

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents from the knowledge base")
for doc in documents:
    print(f"  - {doc.metadata.get('source', 'unknown')} ({doc.metadata['doc_type']})")

In [ ]:
# 我们来看一份文档
# Let's peek at one document

print(documents[0].page_content[:500])

## 步骤 2：将文档分块

我们将文档分成更小的块，以便矢量搜索可以找到最相关的位。
使用 LangChain 的“RecursiveCharacterTextSplitter”——与课程相同。

In [ ]:
# 分成块
# Split into chunks

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(f"\nFirst chunk preview:\n{chunks[0].page_content[:300]}...")

## 步骤 3：创建嵌入并存储在 Chroma 中

使用 HuggingFace 的“all-MiniLM-L6-v2”模型 - 它是免费的并且效果很好。
将所有内容存储在色度矢量数据库中。

In [ ]:
# 创建嵌入并存储在 Chroma 中
# Create embeddings and store in Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 删除旧数据库（如果存在）
# Delete old database if it exists
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

print(f"Vector store created with {vectorstore._collection.count()} documents!")

## 步骤 4：构建 RAG 管道

现在我们连接一切：
1. 用户提出问题
2. 我们在向量存储中搜索相关块
3.我们将chunks+问题传递给LLM
4. LLM给出了有根据的答案

In [ ]:
# 设置检索器和 LLM
# Set up retriever and LLM

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
SYSTEM_PROMPT = """
You are the NaijaCode Assistant — a helpful, friendly chatbot for employees of NaijaCode, a Nigerian tech company.
You answer questions about NaijaCode's employees, products, and company policies.

Rules:
1. Use the provided context to answer questions accurately.
2. Be friendly and professional. You can throw in a little Nigerian flavor if appropriate.
3. If you don't know the answer or it's not in the context, say so honestly.
4. Give concise but complete answers.

Context from the NaijaCode knowledge base:
{context}
"""

In [ ]:
def answer_question(question, history):
    """RAG pipeline: retrieve context, then generate answer"""
    
    # 检索相关块
    # Retrieve relevant chunks
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    
    # 构建带有上下文的提示
    # Build the prompt with context
    system_prompt = SYSTEM_PROMPT.format(context=context)
    
    # 致电法学硕士
    # Call the LLM
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])
    
    return response.content

## 让我们测试一下！

In [ ]:
# 测试一些问题
# Test some questions

test_questions = [
    "Who is the CTO of NaijaCode?",
    "What is the CodeNaija IDE?",
    "How many days of annual leave do employees get?",
    "Who won the TechHer Nigeria award?",
    "What is the pricing for NaijaCode Marketplace templates?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q, [])}")
    print("-" * 80)

## 步骤5：Gradio聊天界面

是时候用合适的聊天 UI 让它看起来更清晰了！

In [ ]:
gr.ChatInterface(
    fn=answer_question,
    title="🇳🇬 NaijaCode Assistant",
    description="Ask me anything about NaijaCode — our employees, products, and policies!",
    examples=[
        "Who founded NaijaCode?",
        "What products does NaijaCode offer?",
        "Tell me about the engineering team",
        "What is the leave policy?",
        "How much funding has NaijaCode raised?",
    ],
    type="messages",
).launch(inbrowser=True)

## 就是这样！ 🎉

我们从头开始构建了完整的 RAG 管道：
1. ✅ 从我们的自定义知识库加载文档
2. ✅ 将它们分块以便高效检索
3. ✅ 使用 HuggingFace 创建向量嵌入
4. ✅ 将它们存储在 Chroma 中
5. ✅ 使用 Gradio 建立 RAG 聊天

### 扩展这个的想法：
- 添加更多知识库文件（合同、常见问题解答等）
- 尝试不同的嵌入模型（OpenAI、Cohere）
- 添加重新排名，就像我们在第 5 天所做的那样
- 构建评估测试来衡量准确性
- 为助手添加洋泾浜英语模式 😄